# Refua -> refua-clinical object API end-to-end

This notebook uses the new object-oriented clinical API. It starts from a small molecule and target, builds Refua-derived payload context, and runs simulation + management artifacts from a single fluent workflow.

In [ ]:
from __future__ import annotations

from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display


def _add_src(path: Path) -> None:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))


cwd = Path.cwd().resolve()
for base in (cwd, cwd.parent, cwd.parent.parent):
    _add_src(base / "src")
    _add_src(base / "refua" / "src")
    _add_src(base / "refua-clinical" / "src")


from refua import Protein, SM
from refua_clinical import ClinicalStudy

## 1) Define a molecule and target in Refua

In [ ]:
smiles = "Cn1cnc2n(C)c(=O)n(C)c(=O)c12"
target_name = "Demo target"
target_sequence = "MSEQNNTEMTFQIQRIYTKDISFEAPNAPHVFQQLAGKYTPEEIRNVLSTLQKAD"

molecule = SM(smiles, lazy=True)
target = Protein(target_sequence, ids="A")

molecule_props = {
    "mol_wt": float(molecule.mol_wt()),
    "mol_log_p": float(molecule.logp()),
    "tpsa": float(molecule.tpsa()),
    "num_h_donors": float(molecule.hbd()),
    "num_h_acceptors": float(molecule.hba()),
    "num_rotatable_bonds": float(molecule.num_rotatable_bonds()),
    "qed": float(molecule.qed()),
    "medchem_alert_count": float(molecule.medchem_alert_count() or 0.0),
}

target_props = {
    "length": float(target.length()),
    "gravy": float(target.gravy()),
    "instability_index": float(target.instability_index()),
    "antibody_liability_score": float(target.antibody_liability_score()),
}

display(pd.Series(molecule_props, name="molecule").to_frame())
display(pd.Series(target_props, name="target").to_frame())

## 2) Build Refua payload context (ADMET + affinity + confidence + properties)

In [ ]:
def _heuristic_admet(smiles_value: str, props: dict[str, float]) -> dict[str, object]:
    bio = max(0.05, min(0.95, 0.85 - max(props["mol_wt"] - 450.0, 0.0) / 500.0))
    safety = max(0.05, min(0.95, 0.80 - 0.08 * props["medchem_alert_count"] - 0.05 * max(props["mol_log_p"] - 3.5, 0.0)))
    adme = max(0.05, min(0.95, 0.80 - max(props["tpsa"] - 120.0, 0.0) / 220.0))
    admet = float((bio + safety + adme) / 3.0)
    red_flags = []
    if safety < 0.40:
        red_flags.append("hERG")
    if props["medchem_alert_count"] >= 2:
        red_flags.append("DILI")
    return {
        "smiles": smiles_value,
        "admet_score": admet,
        "adme_score": float(adme),
        "safety_score": float(safety),
        "red_flags": red_flags,
        "yellow_flags": ["HeuristicProfile"],
        "num_predictions": 3,
        "scores": {
            "score_Bioavailability_Ma": float(bio),
            "score_hERG": float(max(0.05, min(0.95, safety))),
            "score_DILI": float(max(0.05, min(0.95, safety - 0.03))),
            "score_admet": admet,
        },
    }


try:
    admet_profile = molecule.admet_profile(model_variant="9b-chat", include_scoring=True)
    admet_source = "model"
except Exception as exc:
    admet_profile = _heuristic_admet(smiles, molecule_props)
    admet_source = f"heuristic ({type(exc).__name__})"

affinity = {
    "ic50": float(max(5.0, min(300.0, 80.0 - 35.0 * molecule_props["qed"]))),
    "binding_probability": float(max(0.05, min(0.95, 0.45 + 0.5 * molecule_props["qed"]))),
}

structure = {
    "confidence_score": float(max(0.40, min(0.95, 0.65 + 0.2 * molecule_props["qed"]))),
}

refua_payload = {
    "ligands": [
        {
            "ligand_id": "lead_a",
            "smiles": smiles,
            "rdkit": molecule_props,
            "admet": admet_profile,
            "affinity": affinity,
            "structure": structure,
        }
    ],
    "target_properties": target_props,
}

print("ADMET source:", admet_source)
display(pd.Series({"ic50": affinity["ic50"], "binding_probability": affinity["binding_probability"], "confidence": structure["confidence_score"]}, name="lead_signals").to_frame())

## 3) Build and run clinical simulation with object API

In [ ]:
study = (
    ClinicalStudy.default()
    .trial(
        trial_id="refua-object-api-e2e",
        indication=f"{target_name}-aligned indication",
        phase="Phase II",
        objective="Run Refua-informed clinical simulation with management outputs.",
        seed=37,
        replicates=72,
    )
    .refua_payload(refua_payload, apply=True, max_candidate_arms=3)
)

run = study.simulate()
display(pd.Series(run.summary, name="run_summary").to_frame())

## 4) Protocol, optimization, VOI, and advice

In [ ]:
protocol = run.recommend_protocol(
    replicates_per_candidate=30,
    candidate_total_n=[140, 180, 220],
    candidate_interims=[20, 30, 45],
)
optimization = run.optimize(
    replicates_per_candidate=30,
    candidate_total_n=[140, 180, 220],
    candidate_interims=[20, 30, 45],
)
voi = run.value_of_information(extra_n=[0, 30, 60], replicates_per_scenario=30)
advice = run.advise(
    protocol=protocol,
    optimization=optimization,
    voi=voi,
    include_sensitivity=True,
    sensitivity_replicates=20,
)

display(pd.Series(protocol.protocol["simulated_performance"], name="protocol_performance").to_frame())
display(pd.DataFrame(optimization.payload["pareto_front"]).head())
display(pd.DataFrame(voi.payload["scenarios"]).head())
display(pd.DataFrame(advice.report["recommendations"]).head())

## 5) Replicate-level clinical data and artifact export

In [ ]:
replicates = pd.DataFrame(run.payload["replicates"])[
    [
        "replicate_id",
        "treatment_effect",
        "p_value",
        "achieved_target",
        "safety_event_rate",
        "enrolled_n",
        "stop_reason",
    ]
]
display(replicates.head(10))

workup = run.workup(
    replicates_per_candidate=30,
    candidate_total_n=[140, 180, 220],
    candidate_interims=[20, 30, 45],
    voi_extra_n=[0, 30, 60],
    voi_replicates_per_scenario=30,
    include_sensitivity=True,
    sensitivity_replicates=20,
)
manifest = workup.save("artifacts/notebook_e2e_object_api")
print(json.dumps(manifest, indent=2))